In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
import importlib.util
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import gymnasium as gym
from gymnasium.wrappers import TimeLimit

# Load UnbalancedDisk.py directly — bypasses __init__.py which imports usb (hardware only)
_disk_path = os.path.join('..', 'gym_unbalanced_disk', 'envs', 'UnbalancedDisk.py')
_spec = importlib.util.spec_from_file_location('UnbalancedDisk', _disk_path)
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
UnbalancedDisk_sincos = _mod.UnbalancedDisk_sincos

# Patch reset() to accept modern gymnasium signature (seed, options)
class UnbalancedDiskSinCosFixed(UnbalancedDisk_sincos):
    def reset(self, seed=None, options=None):
        obs = super().reset()
        return (obs if isinstance(obs, tuple) else (obs, {}))

env = TimeLimit(UnbalancedDiskSinCosFixed(), max_episode_steps=500)

env.env.reward_fun = lambda self: np.exp(
    -((self.th % (2*np.pi) - np.pi)**2) / (2*(np.pi/6)**2)
)

print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)

Observation space: Box([ -1.  -1. -40.], [ 1.  1. 40.], (3,), float32)
Action space:      Box(-3.0, 3.0, (), float32)


## Actor Network (Gaussian policy)

In [3]:
class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.mean_head    = nn.Linear(hidden, act_dim)
        self.log_std_head = nn.Linear(hidden, act_dim)

    def forward(self, x):
        h = self.net(x)
        mean    = self.mean_head(h)
        log_std = self.log_std_head(h).clamp(-2, 2)
        return mean, log_std

    def get_action(self, obs):
        mean, log_std = self(obs)
        std  = log_std.exp()
        dist = torch.distributions.Normal(mean, std)
        action   = dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        entropy  = dist.entropy().sum(-1)
        action   = action.clamp(-3., 3.)
        return action, log_prob, entropy

## Critic Network (state value estimator)

In [5]:
class Critic(nn.Module):
    def __init__(self, obs_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

## A2C Training Loop

In [6]:
def train_a2c(env, actor, critic, n_episodes=2000, gamma=0.99,
              lr_actor=3e-4, lr_critic=1e-3, entropy_coef=0.01):

    opt_actor  = torch.optim.Adam(actor.parameters(),  lr=lr_actor)
    opt_critic = torch.optim.Adam(critic.parameters(), lr=lr_critic)

    returns_log = []

    for ep in range(n_episodes):
        obs, _ = env.reset()
        done   = False
        log_probs, values, rewards, entropies = [], [], [], []

        while not done:
            obs_t              = torch.tensor(obs, dtype=torch.float32)
            action, log_prob, entropy = actor.get_action(obs_t)
            value              = critic(obs_t)

            obs_next, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated

            log_probs.append(log_prob)
            values.append(value)
            rewards.append(reward)
            entropies.append(entropy)

            obs = obs_next

        # Compute discounted returns
        G, returns = 0, []
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)

        returns    = torch.tensor(returns,  dtype=torch.float32)
        values     = torch.stack(values)
        log_probs  = torch.stack(log_probs)
        entropies  = torch.stack(entropies)

        advantages = returns - values.detach()
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        actor_loss  = -(log_probs * advantages).mean() - entropy_coef * entropies.mean()
        critic_loss = nn.functional.mse_loss(values, returns)

        opt_actor.zero_grad();  actor_loss.backward();  opt_actor.step()
        opt_critic.zero_grad(); critic_loss.backward(); opt_critic.step()

        returns_log.append(sum(rewards))
        if ep % 100 == 0:
            print(f"Ep {ep:4d} | Avg return (last 100): {np.mean(returns_log[-100:]):.2f}")

    return returns_log

## Run Training

In [ ]:
obs_dim = env.observation_space.shape[0]   # 3: [sin(θ), cos(θ), ω]
act_dim = 1                                # single continuous action

actor  = Actor(obs_dim, act_dim)
critic = Critic(obs_dim)

returns_log = train_a2c(env, actor, critic, n_episodes=2000,
                         gamma=0.99, lr_actor=3e-4, lr_critic=1e-3,
                         entropy_coef=0.01)

# Plot learning curve
plt.figure(figsize=(10, 4))
plt.plot(returns_log, alpha=0.3, color='steelblue', label='episode return')
window = 50
plt.plot(np.convolve(returns_log, np.ones(window)/window, mode='valid'),
         color='steelblue', lw=2, label=f'{window}-ep moving avg')
plt.xlabel("Episode"); plt.ylabel("Total return")
plt.title("A2C on Unbalanced Disc — learning curve")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Ep    0 | Avg return (last 100): 0.00
Ep  100 | Avg return (last 100): 50.85
Ep  200 | Avg return (last 100): 109.40
